# Hướng dẫn sử dụng:
1. Chạy Ô Số 1 để kết nối với Google Drive của bạn.
2. Chạy Ô Số 2 để giải nén dữ liệu đã qua tiền xử lý (48 điểm, chuẩn hóa).
3. Chạy Ô Số 3 để Train Mô Hình GRU. Mô hình sẽ tự động lưu lại vào trong Drive của bạn.

In [ ]:
# Ô Số 1: KẾT NỐI GOOGLE DRIVE
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Ô Số 2: GIẢI NÉN DỮ LIỆU
import os
!unzip -q -o /content/drive/MyDrive/dataset_keypoints.zip -d /content/data/keypoints_splited
print("✅ Giải nén thành công toàn bộ dữ liệu!")


In [ ]:
# Ô Số 3: TIẾN HÀNH TRAIN MÔ HÌNH VỚI GPU
import os
import json
import glob
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Kiểm tra GPU
print("GPU đang sử dụng:", tf.test.gpu_device_name())

DATA_DIR = '/content/data/keypoints_splited'
MAX_FRAMES = 60
NUM_POINTS = 48
NUM_DIMS = 3
BATCH_SIZE = 64
EPOCHS = 100

MODELS_DIR = '/content/drive/MyDrive/models_vsl'
CLASSES_FILE = '/content/drive/MyDrive/models_vsl/classes.json'

def get_class_mapping():
    train_dir = os.path.join(DATA_DIR, 'train')
    class_names = sorted([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))])
    if not os.path.exists(os.path.dirname(CLASSES_FILE)):
        os.makedirs(os.path.dirname(CLASSES_FILE))
    with open(CLASSES_FILE, 'w', encoding='utf-8') as f:
        json.dump(class_names, f, ensure_ascii=False, indent=4)
    class_to_id = {name: idx for idx, name in enumerate(class_names)}
    return class_to_id, class_names

def pad_or_truncate(data, max_frames):
    # Dữ liệu từ preprocess.py mới đã được resample chuẩn 60, hàm này dùng làm fail-safe
    t = data.shape[0]
    if t > max_frames:
        return data[:max_frames, :, :]
    elif t < max_frames:
        pad_size = max_frames - t
        padding = np.zeros((pad_size, NUM_POINTS, NUM_DIMS), dtype=np.float32)
        return np.vstack([data, padding])
    return data

def data_generator(split='train'):
    class_to_id, _ = get_class_mapping()
    split_dir = os.path.join(DATA_DIR, split)
    pattern = os.path.join(split_dir, '*', '*.npy')
    files = glob.glob(pattern)
    if split == 'train':
        np.random.shuffle(files)
    for file_path in files:
        class_name = os.path.basename(os.path.dirname(file_path))
        label_id = class_to_id[class_name]
        data = np.load(file_path).astype(np.float32)
        data = pad_or_truncate(data, MAX_FRAMES)
        yield data, label_id

def get_dataset(split='train', batch_size=32):
    dataset = tf.data.Dataset.from_generator(
        lambda: data_generator(split),
        output_signature=(
            tf.TensorSpec(shape=(MAX_FRAMES, NUM_POINTS, NUM_DIMS), dtype=tf.float32),
            tf.TensorSpec(shape=(), dtype=tf.int32)
        )
    )
    if split == 'train':
        dataset = dataset.shuffle(buffer_size=1000)
    
    # Làm phẳng dữ liệu cho GRU
    def reshape_fn(x, y):
        batch_size_dyn = tf.shape(x)[0]
        x_reshaped = tf.reshape(x, [batch_size_dyn, MAX_FRAMES, NUM_POINTS * NUM_DIMS])
        return x_reshaped, y
        
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset.map(reshape_fn, num_parallel_calls=tf.data.AUTOTUNE)

def build_gru_model(input_shape, num_classes):
    flattened_shape = (input_shape[0], input_shape[1] * input_shape[2])
    model = Sequential([
        GRU(128, return_sequences=True, activation='tanh', input_shape=flattened_shape),
        Dropout(0.2),
        BatchNormalization(),
        GRU(256, return_sequences=True, activation='tanh'),
        Dropout(0.2),
        BatchNormalization(),
        GRU(128, return_sequences=False, activation='tanh'),
        Dropout(0.2),
        BatchNormalization(),
        Dense(128, activation='relu'),
        Dense(num_classes, activation='softmax')
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss='sparse_categorical_crossentropy',
                  metrics=['accuracy'])
    return model

print("Đang chuẩn bị dữ liệu...")
class_to_id, class_names = get_class_mapping()
num_classes = len(class_names)

train_dataset = get_dataset(split='train', batch_size=BATCH_SIZE)
val_dataset = get_dataset(split='test', batch_size=BATCH_SIZE)

print(f"Khởi tạo mô hình GRU cho {num_classes} từ vựng...")
model = build_gru_model((MAX_FRAMES, NUM_POINTS, NUM_DIMS), num_classes)

checkpoint_path = os.path.join(MODELS_DIR, 'vsl_gru_baseline_colab.keras')
callbacks = [
    ModelCheckpoint(checkpoint_path, monitor='val_accuracy', verbose=1, save_best_only=True, mode='max'),
    EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)
]

print("Bắt đầu huấn luyện...")
model.fit(train_dataset, validation_data=val_dataset, epochs=EPOCHS, callbacks=callbacks)
